In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


Mount Drive & Set Paths

In [19]:
import os, sys

SCGAN_DIR  = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master"
DATA_DIR   = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/OG_scGAN_pbmc_data"       # stored in Colab session (fast, reusable within session)
EXP_DIR    = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/experiments"
PARAM_PATH = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/parameters.json" # Why do we need it?

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXP_DIR,  exist_ok=True)

# Add scGAN to Python path so imports work
sys.path.insert(0, SCGAN_DIR)

print("✅ Paths configured")
print(f"  scGAN code : {SCGAN_DIR}")
print(f"  Data       : {DATA_DIR}")
print(f"  Experiments: {EXP_DIR}")

✅ Paths configured
  scGAN code : /content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master
  Data       : /content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/OG_scGAN_pbmc_data
  Experiments: /content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/experiments


Download Original scGAN Dataset (Run only once, if saved never run again)

In [12]:

# ── Install only what's needed for data loading ───────────────────────────────
!pip install -q scanpy anndata

import scanpy as sc

H5AD_PATH = f"{DATA_DIR}/pbmc_68k.h5ad"

if os.path.exists(H5AD_PATH):
    print(f"✅ Dataset already exists — skipping download")

else:
    # ── Download ──────────────────────────────────────────────────────────────
    TAR_PATH = "/content/pbmc.tar.gz"   # save to /content/ NOT to Drive (no spaces)
    print("Downloading 68k PBMC...")
    !wget -q --show-progress \
        -O "{TAR_PATH}" \
        "http://cf.10xgenomics.com/samples/cell-exp/1.1.0/fresh_68k_pbmc_donor_a/fresh_68k_pbmc_donor_a_filtered_gene_bc_matrices.tar.gz"

    # ── Extract ───────────────────────────────────────────────────────────────
    print("Extracting...")
    !tar -xzf "{TAR_PATH}" -C "{DATA_DIR}"
    !ls "{DATA_DIR}"   # show what was extracted so we can verify the path

    # ── Check extracted folder structure ─────────────────────────────────────
    print("\nFolder contents:")
    for root, dirs, files in os.walk(DATA_DIR):
        level = root.replace(DATA_DIR, '').count(os.sep)
        print('  ' * level + os.path.basename(root) + '/')
        for f in files[:3]:   # show first 3 files per folder
            print('  ' * (level+1) + f)

    # ── Convert mtx → h5ad ───────────────────────────────────────────────────
    MTX_PATH = f"{DATA_DIR}/filtered_matrices_mex/hg19"

    if not os.path.exists(MTX_PATH):
        # list actual extracted path so we can fix it
        print(f"\n⚠️  Expected MTX path not found: {MTX_PATH}")
        print("Actual extracted contents shown above — paste them here")
    else:
        print("\nConverting mtx → h5ad...")
        adata = sc.read_10x_mtx(MTX_PATH, var_names='gene_symbols', cache=True)
        adata.var_names_make_unique()
        adata.write(H5AD_PATH)
        print(f"✅ Saved: {H5AD_PATH}")
        print(f"   Shape: {adata.shape}")

/content/pbmc.tar.g 100%[===================>] 118.68M   303MB/s    in 0.4s    
Extracting...
filtered_matrices_mex

Folder contents:
OG_scGAN_pbmc_data/
  filtered_matrices_mex/
    hg19/
      barcodes.tsv
      genes.tsv
      matrix.mtx

Converting mtx → h5ad...
✅ Saved: /content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/OG_scGAN_pbmc_data/pbmc_68k.h5ad
   Shape: (68579, 32738)


Convert mtx to h5ad(authors preferred format) - but why?

In [13]:
import scanpy as sc
import os

MTX_PATH = f"{DATA_DIR}/filtered_matrices_mex/hg19"   # path after extraction
OUT_H5AD = f"{DATA_DIR}/pbmc_68k.h5ad"

adata = sc.read_10x_mtx(
    MTX_PATH,
    var_names='gene_symbols',
    cache=True
)

adata.var_names_make_unique()
print(f"Raw shape: {adata.shape}")   # should be ~68k cells x 32k genes
adata.write(OUT_H5AD)
print(f"✅ Saved to {OUT_H5AD}")

Raw shape: (68579, 32738)
✅ Saved to /content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/OG_scGAN_pbmc_data/pbmc_68k.h5ad


Build parameters.json


In [21]:
import json

#EXP_DIR  = f"{ROOT}/experiments"
os.makedirs(EXP_DIR, exist_ok=True)

params = {
    "exp_param": {
        "experiments_dir": EXP_DIR,
        "GPU": [0]
    },
    "experiments": {
        "pbmc_cscGAN": {
            "input_ds": {
                "raw_input": OUT_H5AD,
                "filtering": {
                    "min_cells": 3,      # remove genes in <3 cells
                    "min_genes": 10      # remove cells with <10 genes
                },
                "scale": "normalize_per_cell_LS_20000",   # their exact normalization
                "clustering": {
                    "res": 0.15          # produces 10 clusters as in paper
                },
                "split": {
                    "test_cells"     : 0,
                    "valid_cells"    : 2000,
                    "balanced_split" : True,
                    "split_seed"     : "default"
                }
            },
            "training": {
                "max_steps"      : 1000000,
                "batch_size"     : 128,
                "critic_iters"   : 5,       # WGAN critic steps per generator step
                "checkpoint"     : None,
                "progress_freq"  : 10,
                "validation_freq": 1000,
                "save_freq"      : 1000,
                "summary_freq"   : 50,
                "learning_rate"  : {
                    "decay"      : True,
                    "alpha_0"    : 0.0001,
                    "alpha_final": 0.00001
                },
                "optimizer": {
                    "algorithm": "AMSGrad",
                    "beta1": 0.5,
                    "beta2": 0.9
                }
            },
            "model": {
                "type"           : "cscGAN",
                "latent_dim"     : 128,
                "output_LSN"     : 20000,
                "critic_layers"  : [1024, 512, 256],   # discriminator architecture
                "gen_layers"     : [256, 512, 1024],   # generator architecture
                "critic_cond_type": "proj",             # projection discriminator
                "gen_cond_type"  : "batchnorm",         # conditional batch norm
                "lambd"          : 10                   # gradient penalty lambda
            }
        }
    }
}

#PARAM_PATH = f"{ROOT}/parameters.json"
with open(PARAM_PATH, 'w') as f:
    json.dump(params, f, indent=4)

print("✅ parameters.json written")
print(json.dumps(params, indent=2))

✅ parameters.json written
{
  "exp_param": {
    "experiments_dir": "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/experiments",
    "GPU": [
      0
    ]
  },
  "experiments": {
    "pbmc_cscGAN": {
      "input_ds": {
        "raw_input": "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/OG_scGAN_pbmc_data/pbmc_68k.h5ad",
        "filtering": {
          "min_cells": 3,
          "min_genes": 10
        },
        "scale": "normalize_per_cell_LS_20000",
        "clustering": {
          "res": 0.15
        },
        "split": {
          "test_cells": 0,
          "valid_cells": 2000,
          "balanced_split": true,
          "split_seed": "default"
        }
      },
      "training": {
        "max_steps": 1000000,
        "batch_size": 128,
        "critic_iters": 5,
        "checkpoint": null,
        "progress_freq": 10,
        "validation_freq": 1000,
        "save_freq": 1000,
        "summary_freq": 50,
        "learning_rate": {
        

Preprocess Data

In [26]:
import os
os.chdir(SCGAN_DIR)

!python main.py --param {PARAM_PATH} --process

print("✅ Preprocessing done")

2026-05-21 19:08:53.023311: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-21 19:08:53.083446: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if anndata.__version__ != '0+unknown':
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:28: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if anndata.

To check if the appropriate folders are there or not

In [2]:
import os

SCGAN = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master"

for root, dirs, files in os.walk(SCGAN):
    level = root.replace(SCGAN, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

scGAN-master/
  .dockerignore
  LICENSE
  README.md
  .gitignore
  main.py
  parameters.json
  requirements.txt
  __init__.py
  dockerfile/
    Dockerfile
  estimators/
    AMSGrad.py
    utilities.py
    cscGAN.py
    scGAN.py
    __init__.py
    run_exp.py
  preprocessing/
    __init__.py
    write_tfrecords.py
    process_raw.py
  Misc/
    cond_t-SNE.jpg
    Datasets format conversion.ipynb
    Tensorboard.png
    non-cond_t-SNE.jpg


A way to read everything in the code

In [27]:
# Run this in Colab — prints all files at once
import os

SCGAN_DIR = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master"

files = [
    "estimators/utilities.py",
    "estimators/cscGAN.py",
    "estimators/scGAN.py",
    "estimators/run_exp.py",
    "estimators/AMSGrad.py",
    "preprocessing/process_raw.py",
    "preprocessing/write_tfrecords.py",
]

for f in files:
    full = os.path.join(SCGAN_DIR, f)
    print(f"\n{'='*60}")
    print(f"FILE: {f}")
    print('='*60)
    with open(full, 'r') as fh:
        print(fh.read())


FILE: estimators/utilities.py
#!/usr/bin/python
# -*- coding: utf-8 -*-
"""
A number of classes (Critic and Generator) and helper functions used across all types of models.
"""
import tensorflow as tf
import numpy as np
import sys
import pandas as pd
import scanpy as sc
from estimators.AMSGrad import AMSGrad
from tensorflow.contrib.layers.python import layers
import scipy.sparse as sp_sparse
from natsort import natsorted
from MulticoreTSNE import MulticoreTSNE as TSNE
tsne = TSNE(n_jobs=8)


class Generator:
    """
    Generic class for the Generator network.
    """

    def __init__(self, fake_outputs, batch_size, latent_dim,
                 output_cells_dim, var_scope,  gen_layers,
                 output_lsn, gen_cond_type=None, is_training=None,
                 clusters_ratios=None, clusters_no=None,
                 input_clusters=None, reuse=None):
        """
        Default constructor.

        Parameters
        ----------
        fake_outputs : Tensor
            Tensor

Writing all the 7 required files

In [8]:
import os

SCGAN_DIR = "/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master"

# ═══════════════════════════════════════════════════════════════
# 1. AMSGrad.py
# ═══════════════════════════════════════════════════════════════
amsgrad = '''#!/usr/bin/python
# -*- coding: utf-8 -*-
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from tensorflow.python.training import optimizer as optimizer_lib


class AMSGrad(optimizer_lib.Optimizer):
    def __init__(self, learning_rate=0.01, beta1=0.9, beta2=0.99,
                 epsilon=1e-8, use_locking=False, name="AMSGrad"):
        super(AMSGrad, self).__init__(use_locking, name)
        self._lr        = learning_rate
        self._beta1     = beta1
        self._beta2     = beta2
        self._epsilon   = epsilon
        self._lr_t      = None
        self._beta1_t   = None
        self._beta2_t   = None
        self._epsilon_t = None
        self._beta1_power = None
        self._beta2_power = None

    def _create_slots(self, var_list):
        first_var = min(var_list, key=lambda x: x.name)
        create_new = self._beta1_power is None
        if not create_new:
            try:
                create_new = (self._beta1_power.graph is not first_var.graph)
            except Exception:
                create_new = True
        if create_new:
            with tf.colocate_with(first_var):
                self._beta1_power = tf.Variable(self._beta1, name="beta1_power", trainable=False)
                self._beta2_power = tf.Variable(self._beta2, name="beta2_power", trainable=False)
        for v in var_list:
            self._zeros_slot(v, "m",    self._name)
            self._zeros_slot(v, "v",    self._name)
            self._zeros_slot(v, "vhat", self._name)

    def _prepare(self):
        self._lr_t      = tf.convert_to_tensor(self._lr)
        self._beta1_t   = tf.convert_to_tensor(self._beta1)
        self._beta2_t   = tf.convert_to_tensor(self._beta2)
        self._epsilon_t = tf.convert_to_tensor(self._epsilon)

    def _apply_dense(self, grad, var):
        beta1_power = tf.cast(self._beta1_power, var.dtype.base_dtype)
        beta2_power = tf.cast(self._beta2_power, var.dtype.base_dtype)
        lr_t      = tf.cast(self._lr_t,      var.dtype.base_dtype)
        beta1_t   = tf.cast(self._beta1_t,   var.dtype.base_dtype)
        beta2_t   = tf.cast(self._beta2_t,   var.dtype.base_dtype)
        epsilon_t = tf.cast(self._epsilon_t, var.dtype.base_dtype)
        lr = lr_t * tf.sqrt(1 - beta2_power) / (1 - beta1_power)
        m = self.get_slot(var, "m")
        m_t = tf.assign(m, beta1_t * m + (1 - beta1_t) * grad, use_locking=self._use_locking)
        v = self.get_slot(var, "v")
        v_t = tf.assign(v, beta2_t * v + (1 - beta2_t) * grad * grad, use_locking=self._use_locking)
        vhat   = self.get_slot(var, "vhat")
        vhat_t = tf.assign(vhat, tf.maximum(v_t, vhat))
        var_update = tf.assign_sub(var, lr * m_t / (tf.sqrt(vhat_t) + epsilon_t), use_locking=self._use_locking)
        return tf.group(*[var_update, m_t, v_t, vhat_t])

    def _resource_apply_dense(self, grad, var):
        return self._apply_dense(grad, var)

    def _apply_sparse(self, grad, var):
        return self._apply_dense(tf.convert_to_tensor(grad), var)

    def _resource_apply_sparse(self, grad, var, indices):
        return self._apply_dense(tf.convert_to_tensor(grad), var)

    def _finish(self, update_ops, name_scope):
        with tf.control_dependencies(update_ops):
            with tf.colocate_with(self._beta1_power):
                upd1 = self._beta1_power.assign(self._beta1_power * self._beta1_t, use_locking=self._use_locking)
                upd2 = self._beta2_power.assign(self._beta2_power * self._beta2_t, use_locking=self._use_locking)
        return tf.group(*update_ops + [upd1, upd2], name=name_scope)
'''

# ═══════════════════════════════════════════════════════════════
# 2. utilities.py
# ═══════════════════════════════════════════════════════════════
utilities = '''#!/usr/bin/python
# -*- coding: utf-8 -*-
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np
import sys
import pandas as pd
import scanpy as sc
from estimators.AMSGrad import AMSGrad
import scipy.sparse as sp_sparse
from natsort import natsorted
from sklearn.manifold import TSNE as skTSNE
tsne = skTSNE(n_components=2, perplexity=30, random_state=42)

# ── layer helpers (replaces tensorflow.contrib.layers) ────────────────────────
def _dense(x, units, activation=None,
           kernel_init='glorot_uniform', bias_init='zeros', use_bias=True):
    return tf.layers.dense(x, units,
                           activation=activation,
                           kernel_initializer=kernel_init,
                           bias_initializer=bias_init,
                           use_bias=use_bias)

def linear(x, units):
    return _dense(x, units, activation=None,
                  kernel_init=tf.glorot_uniform_initializer(), use_bias=False)

def relu_layer(x, units):
    return _dense(x, units, activation=tf.nn.relu,
                  kernel_init=tf.variance_scaling_initializer(mode="fan_avg"),
                  bias_init=tf.zeros_initializer())

def fan_avg_dense(x, units):
    return _dense(x, units, activation=None,
                  kernel_init=tf.variance_scaling_initializer(mode="fan_avg"),
                  bias_init=tf.zeros_initializer())


class Generator:
    def __init__(self, fake_outputs, batch_size, latent_dim,
                 output_cells_dim, var_scope, gen_layers,
                 output_lsn, gen_cond_type=None, is_training=None,
                 clusters_ratios=None, clusters_no=None,
                 input_clusters=None, reuse=None):
        self.batch_size       = batch_size
        self.is_training      = is_training
        self.latent_dim       = latent_dim
        self.output_cells_dim = output_cells_dim
        self.var_scope        = var_scope
        self.gen_layers       = gen_layers
        self.output_lsn       = output_lsn
        self.gen_cond_type    = gen_cond_type
        self.clusters_no      = clusters_no
        self.input_clusters   = input_clusters
        self.reuse            = reuse
        self.fake_outputs     = fake_outputs
        self.clusters_ratios  = clusters_ratios

    @classmethod
    def create_cond_generator(cls, z_input, batch_size, latent_dim,
                              output_cells_dim, var_scope, gen_layers,
                              output_lsn, gen_cond_type, clusters_ratios,
                              is_training, clusters_no=None,
                              input_clusters=None, reuse=None):
        with tf.variable_scope(var_scope, reuse=reuse):
            for i_lay, size in enumerate(gen_layers):
                with tf.variable_scope("generator_layers_" + str(i_lay + 1)):
                    z_input = linear(z_input, size)
                    if gen_cond_type == "batchnorm":
                        z_input = batchnorm([0], z_input, is_training=is_training,
                                            labels=input_clusters, n_labels=clusters_no)
                    elif gen_cond_type == "layernorm":
                        z_input = layernorm([1], z_input,
                                            labels=input_clusters, n_labels=clusters_no)
                    z_input = tf.nn.relu(z_input)
            with tf.variable_scope("generator_layers_output"):
                fake_outputs = fan_avg_dense(z_input, output_cells_dim)
                fake_outputs = tf.nn.relu(fake_outputs)
                if output_lsn is not None:
                    gammas_output = tf.Variable(
                        np.ones(z_input.shape.as_list()[0]) * output_lsn, trainable=False)
                    sigmas   = tf.reduce_sum(fake_outputs, axis=1)
                    scale_ls = tf.cast(gammas_output, dtype=tf.float32) / (sigmas + sys.float_info.epsilon)
                    fake_outputs = tf.transpose(tf.transpose(fake_outputs) * scale_ls)
        return cls(fake_outputs, batch_size, latent_dim, output_cells_dim,
                   var_scope, gen_layers, output_lsn, gen_cond_type=gen_cond_type,
                   is_training=is_training, clusters_ratios=clusters_ratios,
                   clusters_no=clusters_no, input_clusters=input_clusters, reuse=reuse)

    @classmethod
    def create_generator(cls, z_input, batch_size, latent_dim,
                         output_cells_dim, var_scope, gen_layers,
                         output_lsn, is_training, reuse=None):
        with tf.variable_scope(var_scope, reuse=reuse):
            for i_lay, size in enumerate(gen_layers):
                with tf.variable_scope("generator_layers_" + str(i_lay + 1)):
                    z_input = linear(z_input, size)
                    z_input = batchnorm([0], z_input, is_training)
                    z_input = tf.nn.relu(z_input)
            with tf.variable_scope("generator_layers_output"):
                fake_outputs = fan_avg_dense(z_input, output_cells_dim)
                fake_outputs = tf.nn.relu(fake_outputs)
                if output_lsn is not None:
                    gammas_output = tf.Variable(
                        np.ones(z_input.shape.as_list()[0]) * output_lsn, trainable=False)
                    sigmas   = tf.reduce_sum(fake_outputs, axis=1)
                    scale_ls = tf.cast(gammas_output, dtype=tf.float32) / (sigmas + sys.float_info.epsilon)
                    fake_outputs = tf.transpose(tf.transpose(fake_outputs) * scale_ls)
        return cls(fake_outputs, batch_size, latent_dim, output_cells_dim,
                   var_scope, gen_layers, output_lsn, is_training=is_training, reuse=reuse)


class Critic:
    def __init__(self, xinput, dist, var_scope, critic_layers,
                 input_clusters=None, clusters_no=None, reuse=None):
        self.xinput         = xinput
        self.var_scope      = var_scope
        self.critic_layers  = critic_layers
        self.clusters_no    = clusters_no
        self.input_clusters = input_clusters
        self.reuse          = reuse
        self.dist           = dist

    @classmethod
    def create_cond_critic(cls, xinput, input_clusters, var_scope,
                           critic_layers, clusters_no, reuse=None):
        with tf.variable_scope(var_scope, reuse=reuse):
            for i_lay, output_size in enumerate(critic_layers):
                with tf.variable_scope("layers_" + str(i_lay + 1)):
                    xinput = relu_layer(xinput, output_size)
            with tf.variable_scope("layers_proj"):
                proj_weights_m = tf.get_variable("proj_weights_m",
                    [clusters_no, critic_layers[-1], 1], dtype=tf.float32,
                    initializer=tf.glorot_uniform_initializer())
                proj_bias_m = tf.get_variable("proj_bias_m",
                    [clusters_no, 1], dtype=tf.float32,
                    initializer=tf.zeros_initializer())
                proj_weights = tf.nn.embedding_lookup(proj_weights_m, input_clusters)
                proj_bias    = tf.nn.embedding_lookup(proj_bias_m,    input_clusters)
                output = tf.einsum("ij,ijk->ik", xinput, proj_weights)
                dist   = tf.add(output, proj_bias)
        return cls(xinput, dist, var_scope, critic_layers,
                   input_clusters=input_clusters, clusters_no=clusters_no, reuse=reuse)

    @classmethod
    def create_cond_critic_proj(cls, xinput, input_clusters, var_scope,
                                critic_layers, clusters_no, reuse=None):
        with tf.variable_scope(var_scope, reuse=reuse):
            for i_lay, output_size in enumerate(critic_layers):
                with tf.variable_scope("layers_" + str(i_lay + 1)):
                    xinput = relu_layer(xinput, output_size)
            with tf.variable_scope("layers_proj"):
                proj_weights_m = tf.get_variable("proj_weights_m",
                    [clusters_no, critic_layers[-1], 1], dtype=tf.float32,
                    initializer=tf.glorot_uniform_initializer())
                proj_weights  = tf.nn.embedding_lookup(proj_weights_m, input_clusters)
                output_proj   = tf.einsum("ij,ijk->ik", xinput, proj_weights)
            with tf.variable_scope("layers_output"):
                output = linear(xinput, 1)
                dist   = tf.add(output_proj, output)
        return cls(xinput, dist, var_scope, critic_layers,
                   input_clusters=input_clusters, clusters_no=clusters_no, reuse=reuse)

    @classmethod
    def create_critic(cls, xinput, var_scope, critic_layers, reuse=None):
        with tf.variable_scope(var_scope, reuse=reuse):
            for i_lay, output_size in enumerate(critic_layers):
                with tf.variable_scope("layers_" + str(i_lay + 1)):
                    xinput = relu_layer(xinput, output_size)
            with tf.variable_scope("layers_output"):
                output = linear(xinput, 1)
        return cls(xinput, output, var_scope, critic_layers, reuse=reuse)


def add_random_labels(clusters_ratios, batch_size):
    clusters_ratios = tf.cast(clusters_ratios, dtype=tf.float32)
    mn_logits = tf.tile(tf.math.log(clusters_ratios), (batch_size, 1))
    labels = tf.cast(tf.random.categorical(mn_logits, 1), tf.int32)
    return tf.squeeze(labels)

def add_random_input(batch_size, latent_dim):
    return tf.random.normal([batch_size, latent_dim])

def save_generated_cells(fake_cells, file_name, fake_labels=None):
    s_gen_mat = sp_sparse.csr_matrix(fake_cells)
    sc_fake   = sc.AnnData(s_gen_mat)
    if fake_labels is not None:
        groups = fake_labels.astype("U")
        sc_fake.obs["cluster"] = pd.Categorical(
            values=groups, categories=natsorted(np.unique(groups)))
    sc_fake.obs_names = np.repeat("fake", sc_fake.shape[0])
    sc_fake.obs_names_make_unique()
    sc_fake.write(file_name)

def rescale(fake_cells, scaling, scale_value):
    if "normalize_per_cell_LS_" in str(scaling):
        fake_cells = fake_cells * float(scale_value)
    return fake_cells

def sc_summary(name_scope, sc_batch):
    with tf.name_scope(name_scope):
        tf.summary.scalar("mean",      tf.reduce_mean(sc_batch))
        tf.summary.histogram("histogram", sc_batch)

def set_learning_rate(alpha_0, alpha_final, global_step, max_steps):
    return tf.train.exponential_decay(
        learning_rate=alpha_0, global_step=global_step,
        decay_steps=max_steps, decay_rate=alpha_final / alpha_0)

def set_global_step():
    global_step     = tf.train.get_or_create_global_step()
    incr_global_step = tf.assign(global_step, global_step + 1)
    return global_step, incr_global_step

def gradient_step(training_variables, training_loss, learning_rate,
                  beta1, beta2, optimizer="AMSGrad"):
    if optimizer == "AMSGrad":
        opt = AMSGrad(learning_rate=learning_rate, beta1=beta1, beta2=beta2)
    else:
        opt = tf.train.AdamOptimizer(learning_rate=learning_rate, beta1=beta1, beta2=beta2)
    grads_and_vars = opt.compute_gradients(training_loss, var_list=training_variables)
    return opt.apply_gradients(grads_and_vars), grads_and_vars

def batchnorm(axes, inputs, is_training, decay=0.999, labels=None, n_labels=None):
    moving_mean = tf.get_variable("BN_moving_mean", inputs.get_shape().as_list()[1:],
                                  dtype=tf.float32, initializer=tf.zeros_initializer(), trainable=False)
    moving_var  = tf.get_variable("BN_moving_var",  inputs.get_shape().as_list()[1:],
                                  dtype=tf.float32, initializer=tf.ones_initializer(),  trainable=False)
    n_neurons = inputs.get_shape().as_list()[1]
    if labels is None:
        with tf.variable_scope("BLN"):
            offset = tf.get_variable("BN_offset", [n_neurons], dtype=tf.float32, initializer=tf.zeros_initializer())
            scale  = tf.get_variable("BN_scale",  [n_neurons], dtype=tf.float32, initializer=tf.ones_initializer())
    else:
        with tf.variable_scope("BLN"):
            offset_m = tf.get_variable("BN_offset", [n_labels] + inputs.get_shape().as_list()[1:],
                                       dtype=tf.float32, initializer=tf.zeros_initializer())
            scale_m  = tf.get_variable("BN_scale",  [n_labels] + inputs.get_shape().as_list()[1:],
                                       dtype=tf.float32, initializer=tf.ones_initializer())
        offset = tf.squeeze(tf.nn.embedding_lookup(offset_m, labels))
        scale  = tf.squeeze(tf.nn.embedding_lookup(scale_m,  labels))
    def bn_training():
        batch_mean, batch_var = tf.nn.moments(inputs, axes, keepdims=False)
        upd_mean = tf.assign(moving_mean, moving_mean * decay + batch_mean * (1 - decay))
        upd_var  = tf.assign(moving_var,  moving_var  * decay + batch_var  * (1 - decay))
        with tf.control_dependencies([upd_mean, upd_var]):
            return tf.nn.batch_normalization(inputs, batch_mean, batch_var, offset, scale, 1e-5)
    def bn_inference():
        return tf.nn.batch_normalization(inputs, moving_mean, moving_var, offset, scale, 1e-5)
    return tf.cond(is_training, bn_training, bn_inference)

def layernorm(axes, inputs, labels=None, n_labels=None):
    mean, var = tf.nn.moments(inputs, axes, keepdims=True)
    n_neurons = inputs.get_shape().as_list()[axes[0]]
    if labels is None:
        with tf.variable_scope("BLN"):
            offset = tf.get_variable("LN_offset", [n_neurons], dtype=tf.float32, initializer=tf.zeros_initializer())
            scale  = tf.get_variable("LN_scale",  [n_neurons], dtype=tf.float32, initializer=tf.ones_initializer())
    else:
        with tf.variable_scope("BLN"):
            offset_m = tf.get_variable("LN_offset", [n_labels, n_neurons], dtype=tf.float32, initializer=tf.zeros_initializer())
            scale_m  = tf.get_variable("LN_scale",  [n_labels, n_neurons], dtype=tf.float32, initializer=tf.ones_initializer())
        offset = tf.squeeze(tf.nn.embedding_lookup(offset_m, labels))
        scale  = tf.squeeze(tf.nn.embedding_lookup(scale_m,  labels))
    return tf.nn.batch_normalization(inputs, mean, var, offset, scale, 1e-5)
'''

# ═══════════════════════════════════════════════════════════════
# 3. write_tfrecords.py  (only tf.python_io → tf.io)
# ═══════════════════════════════════════════════════════════════
write_tfrecords = open(os.path.join(SCGAN_DIR, "preprocessing/write_tfrecords.py")).read()
write_tfrecords = write_tfrecords.replace(
    "import tensorflow as tf",
    "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
).replace(
    "tf.python_io.TFRecordOptions",    "tf.io.TFRecordOptions"
).replace(
    "tf.python_io.TFRecordCompressionType", "tf.io.TFRecordCompressionType"
).replace(
    "tf.python_io.TFRecordWriter",     "tf.io.TFRecordWriter"
)

# ═══════════════════════════════════════════════════════════════
# 4. process_raw.py  (scanpy API changes)
# ═══════════════════════════════════════════════════════════════
process_raw = open(os.path.join(SCGAN_DIR, "preprocessing/process_raw.py")).read()
process_raw = process_raw.replace(
    "import scanpy as sc",
    "import scanpy as sc\nimport tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
).replace(
    "sc.pp.recipe_zheng17(clustered)",
    """sc.pp.normalize_total(clustered, target_sum=1e4)
            sc.pp.log1p(clustered)
            sc.pp.highly_variable_genes(clustered, n_top_genes=1000)
            clustered = clustered[:, clustered.var.highly_variable]
            sc.pp.scale(clustered, max_value=10)"""
).replace(
    "counts_per_cell_after=lib_size",
    "target_sum=lib_size"
).replace(
    "self.sc_raw.obs.set_value(i, 'split', 'test')",
    "self.sc_raw.obs.loc[i, 'split'] = 'test'"
).replace(
    "self.sc_raw.obs.set_value(i, 'split', 'valid')",
    "self.sc_raw.obs.loc[i, 'split'] = 'valid'"
)

# ═══════════════════════════════════════════════════════════════
# 5. cscGAN.py and scGAN.py — fix tf.python_io + tf.contrib.learn
# ═══════════════════════════════════════════════════════════════
def patch_gan_file(content):
    content = content.replace(
        "import tensorflow as tf",
        "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
    ).replace(
        "from MulticoreTSNE import MulticoreTSNE as TSNE\ntsne = TSNE(n_jobs=20)",
        "from sklearn.manifold import TSNE as skTSNE\ntsne = skTSNE(n_components=2, perplexity=30, random_state=42)"
    ).replace(
        "from MulticoreTSNE import MulticoreTSNE as TSNE\ntsne = TSNE(n_jobs=8)",
        "from sklearn.manifold import TSNE as skTSNE\ntsne = skTSNE(n_components=2, perplexity=30, random_state=42)"
    ).replace(
        "tsne.fit_transform(",
        "tsne.fit_transform("
    ).replace(
        "tf.python_io.TFRecordOptions",      "tf.io.TFRecordOptions"
    ).replace(
        "tf.python_io.TFRecordCompressionType", "tf.io.TFRecordCompressionType"
    ).replace(
        "tf.SparseFeature(",   "tf.io.SparseFeature("
    ).replace(
        "tf.FixedLenFeature(", "tf.io.FixedLenFeature("
    ).replace(
        "tf.sparse_reshape(",        "tf.sparse.reshape("
    ).replace(
        "tf.sparse_tensor_to_dense(","tf.sparse.to_dense("
    ).replace(
        "tf.to_int32(",    "tf.cast(",
    ).replace(
        "tf.random_uniform(", "tf.random.uniform("
    ).replace(
        "tf.random_normal(",  "tf.random.normal("
    ).replace(
        "tf.train.Supervisor(", "tf.compat.v1.train.Supervisor("
    ).replace(
        "tf.train.Saver(",      "tf.compat.v1.train.Saver("
    ).replace(
        "reduction_indices=[1]", "axis=[1]"
    )

    # Fix tf.contrib.learn.read_batch_features → tf.data pipeline
    old_input_fn = """        options = tf.io.TFRecordOptions(
            tf.io.TFRecordCompressionType.GZIP)

        batched_features = tf.contrib.learn.read_batch_features(
            file_pattern=file_paths,
            batch_size=self.batch_size,
            features=feature_map,
            reader=lambda: tf.TFRecordReader(
                options=options),
            num_epochs=epochs)

        sgc = batched_features['scg']

        sparse = tf.sparse.reshape(sgc, (self.batch_size, self.genes_no))

        dense = tf.sparse.to_dense(sparse)

        cluster = tf.squeeze(tf.cast(batched_features['cluster_int']))

        features = tf.reshape(dense, (self.batch_size, self.genes_no))

        return features, cluster"""

    new_input_fn_csc = """        def _parse(serialized):
            parsed = tf.parse_single_example(serialized, features={
                'indices': tf.VarLenFeature(tf.int64),
                'values' : tf.VarLenFeature(tf.float32),
                'cluster_int': tf.FixedLenFeature([1], tf.int64),
            })
            sparse = tf.SparseTensor(
                indices=tf.cast(tf.expand_dims(parsed['indices'].values, 1), tf.int64),
                values=parsed['values'].values,
                dense_shape=[self.genes_no])
            dense   = tf.sparse.to_dense(sparse)
            cluster = tf.cast(tf.squeeze(parsed['cluster_int']), tf.int32)
            return dense, cluster

        compression = "GZIP"
        dataset = tf.data.TFRecordDataset(file_paths, compression_type=compression)
        dataset = dataset.map(_parse).repeat().batch(self.batch_size, drop_remainder=True)
        iterator = tf.data.make_one_shot_iterator(dataset)
        features, cluster = iterator.get_next()
        features = tf.reshape(features, (self.batch_size, self.genes_no))
        return features, cluster"""

    # non-conditional version (scGAN — no cluster)
    old_input_fn_sc = """        options = tf.io.TFRecordOptions(
            tf.io.TFRecordCompressionType.GZIP)

        batched_features = tf.contrib.learn.read_batch_features(
            file_pattern=file_paths,
            batch_size=self.batch_size,
            features=feature_map,
            reader=lambda: tf.TFRecordReader(options=options),
            num_epochs=epochs
            )

        sgc = batched_features['scg']

        sparse = tf.sparse.reshape(sgc, (self.batch_size, self.genes_no))

        dense = tf.sparse.to_dense(sparse)

        features = tf.reshape(dense, (self.batch_size, self.genes_no))

        return features"""

    new_input_fn_sc = """        def _parse(serialized):
            parsed = tf.parse_single_example(serialized, features={
                'indices': tf.VarLenFeature(tf.int64),
                'values' : tf.VarLenFeature(tf.float32),
            })
            sparse = tf.SparseTensor(
                indices=tf.cast(tf.expand_dims(parsed['indices'].values, 1), tf.int64),
                values=parsed['values'].values,
                dense_shape=[self.genes_no])
            dense = tf.sparse.to_dense(sparse)
            return dense

        compression = "GZIP"
        dataset = tf.data.TFRecordDataset(file_paths, compression_type=compression)
        dataset = dataset.map(_parse).repeat().batch(self.batch_size, drop_remainder=True)
        iterator = tf.data.make_one_shot_iterator(dataset)
        features = iterator.get_next()
        features = tf.reshape(features, (self.batch_size, self.genes_no))
        return features"""

    content = content.replace(old_input_fn,    new_input_fn_csc)
    content = content.replace(old_input_fn_sc, new_input_fn_sc)

    # remove the feature_map block that precedes contrib.learn (now handled inside _parse)
    content = content.replace(
        """        feature_map = {'scg': tf.io.SparseFeature(index_key='indices',
                                               value_key='values',
                                               dtype=tf.float32,
                                               size=self.genes_no),
                       'cluster_int': tf.io.FixedLenFeature(1, tf.int64)}

        """, "        ")
    content = content.replace(
        """        feature_map = {'scg': tf.io.SparseFeature(index_key='indices',
                                               value_key='values',
                                               dtype=tf.float32,
                                               size=self.genes_no)
                       }

        """, "        ")
    return content

cscgan_content = open(os.path.join(SCGAN_DIR, "estimators/cscGAN.py")).read()
scgan_content  = open(os.path.join(SCGAN_DIR, "estimators/scGAN.py")).read()
run_exp_content = open(os.path.join(SCGAN_DIR, "estimators/run_exp.py")).read()

cscgan_patched  = patch_gan_file(cscgan_content)
scgan_patched   = patch_gan_file(scgan_content)
run_exp_patched = run_exp_content.replace(
    "import tensorflow as tf",
    "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()"
).replace(
    "tf.reset_default_graph()", "tf.reset_default_graph()"
)

# ═══════════════════════════════════════════════════════════════
# WRITE ALL FILES
# ═══════════════════════════════════════════════════════════════
files_to_write = {
    "estimators/AMSGrad.py"          : amsgrad,
    "estimators/utilities.py"        : utilities,
    "estimators/cscGAN.py"           : cscgan_patched,
    "estimators/scGAN.py"            : scgan_patched,
    "estimators/run_exp.py"          : run_exp_patched,
    "preprocessing/write_tfrecords.py": write_tfrecords,
    "preprocessing/process_raw.py"   : process_raw,
}

for rel_path, content in files_to_write.items():
    fpath = os.path.join(SCGAN_DIR, rel_path)
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"✅ Written: {rel_path}")

print("\n✅ All files patched. Now running preprocessing...")

# ── Run preprocessing ──────────────────────────────────────────
import sys
sys.path.insert(0, SCGAN_DIR)
os.chdir(SCGAN_DIR)
!python main.py --param {PARAM_PATH} --process
print("\n✅ Done")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/scGAN Approach/scGAN-master/preprocessing/write_tfrecords.py'

In [7]:
import os

base = "/content/drive/MyDrive/Ahsan"
for root, dirs, files in os.walk(base):
    for f in files:
        if f == "main.py":
            print(os.path.join(root, f))